# Preparation database "res. 0.01°"

In [ ]:
# Libraries
import os, shutil
import numpy as np
import xarray as xr
import dask.dataframe as dd
from sklearn.model_selection import train_test_split
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute', 
                       cores=24, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface='ib0',
                       walltime='02:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=5)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
# Get data
# Land mask
ds_land = xr.open_zarr(f'{dir01}ds_prep_copernicus_land_mask_r001.zarr')
# Potential training grid cells
ds_pot_train = xr.open_zarr(f'{dir02}ds_pot_train_r001.zarr')
# Potential occurrence grid cells
ds_pot_occur = xr.open_zarr(f'{dir02}ds_pot_occur_r001.zarr')

# Vegetation biomass carbon
ds_cveg_biom = xr.open_zarr(f'{dir02}ds_cveg_biom_r001.zarr')
# Vegetation crop carbon
ds_cveg_crop = xr.open_zarr(f'{dir02}ds_cveg_crop_r001.zarr')
# Vegetation grassland carbon
ds_cveg_gras = xr.open_zarr(f'{dir02}ds_cveg_gras_r001.zarr')
# Vegetation tundra and mountain carbon
ds_cveg_tund = xr.open_zarr(f'{dir02}ds_cveg_tund_r001.zarr')
# Soil carbon
ds_csoil_000_100 = xr.open_zarr(f'{dir02}ds_csoil_000_100_r001.zarr')

# Soilgrids1
ds_soilgrids2017 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2017_r001.zarr')
# Soilgrids2
ds_soilgrids2020 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2020_r001.zarr')
# Worldclim
ds_worldclim = xr.open_zarr(f'{dir01}ds_prep_worldclim_r001.zarr')
# Geom90m
ds_geom90m = xr.open_zarr(f'{dir01}ds_prep_geom90m_r001.zarr')

In [ ]:
# Add prefix "pot_train" to ds_pot_train variable names
ds_pot_train = ds_pot_train \
    .rename({var: f'pot_train_{var}' for var in ds_pot_train.data_vars})

# Add prefix "pot_occur" to ds_pot_occur variable names
ds_pot_occur = ds_pot_occur \
    .rename({var: f'pot_occur_{var}' for var in ds_pot_occur.data_vars})

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List with covariates
vars_soilgrids2017 = ['soilgrids2017_bdricm', 'soilgrids2017_bdrlog',
                      'soilgrids2017_bdticm']

vars_soilgrids2020 = ['soilgrids2020_cec', 'soilgrids2020_cfvo', 
                      'soilgrids2020_clay', 'soilgrids2020_phh2o', 
                      'soilgrids2020_sand', 'soilgrids2020_silt']

vars_worldclim = ['worldclim_bio1', 'worldclim_bio3', 'worldclim_bio4', 
                  'worldclim_bio5', 'worldclim_bio6', 'worldclim_bio12', 
                  'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15', 
                  'worldclim_elev']

vars_geom90m = ['geom90m_convergence', 'geom90m_cti', 'geom90m_eastness',
               'geom90m_northness', 'geom90m_slope', 'geom90m_spi']

list_var_exp = [*vars_soilgrids2017, *vars_soilgrids2020, *vars_worldclim, 
                *vars_geom90m]

---

### Transfrom dataset to dataframe

In [ ]:
# Select relevant variables
ds_soilgrids2017 = ds_soilgrids2017[vars_soilgrids2017]
ds_soilgrids2020 = ds_soilgrids2020[vars_soilgrids2020]
ds_worldclim = ds_worldclim[vars_worldclim]
ds_geom90m = ds_geom90m[vars_geom90m]

# Combine relevant data in one dataset
ds_all = xr.merge([ds_land, ds_pot_train, ds_pot_occur,
                   ds_cveg_biom, ds_cveg_crop, ds_cveg_gras, ds_cveg_tund, 
                   ds_csoil_000_100, ds_soilgrids2017, ds_soilgrids2020, 
                   ds_worldclim, ds_geom90m])

ds_all

In [ ]:
%%time
# Create dataframe with all relevant data
# Exclude non land grid cells
def ds2df_sel(latmin, latmax):

    """Helper function: Transform dataset to dataframe within latmin and latmax"""
    
    # Select area
    ds_all_sel = ds_all \
        .sel(lat=slice(latmax, latmin))
    
    # Transform to dask dataframe
    df_all_sel = ds_all_sel \
        .to_dask_dataframe(dim_order=['lat', 'lon']) \
        .persist()
    
    # Extract land grid cells
    df_all_land_sel = df_all_sel[df_all_sel.copernicus_land_mask == 1]
    
    # Export
    df_all_land_sel  \
        .persist() \
        .repartition(partition_size="200MiB") \
        .to_parquet(f'{dir02}dd_base_latmin{latmin}_latmax{latmax}.parquet')


#remove database file if exists
if os.path.exists(f'{dir02}dd_base_interm'):
    shutil.rmtree(f'{dir02}dd_base_interm')

# dataset needs to be split in tiles because of size
# transfrom dataset tiles to dataframes
# tiles in 10° lat stripes
for latmin, latmax in zip(list(range(70, -61, -10)), 
                          list(range(80, -51, -10))):
    ds2df_sel(latmin, latmax)

# read dataframe tile
list_df_all_vars_land_sel = [dd.read_parquet(dir02 + i) for i in 
                             os.listdir(dir02)
                             if i.startswith('dd_base_latmin')]

# concatenate dataframe parts
dd_base = dd.concat(list_df_all_vars_land_sel)

# adjust dtype of certain columns
dd_base['copernicus_land_mask'] = dd_base['copernicus_land_mask'].astype('bool')
dd_base['cveg_biom'] = dd_base['cveg_biom'].round(0).astype('int16')
dd_base['cveg_crop'] = dd_base['cveg_crop'].round(0).astype('int16')
dd_base['cveg_gras'] = dd_base['cveg_gras'].round(0).astype('int16')
dd_base['cveg_tund'] = dd_base['cveg_tund'].round(0).astype('int16')
dd_base['csoil_000_100'] = dd_base['csoil_000_100'].round(0).astype('int16')

# export
dd_base \
    .persist() \
    .repartition(partition_size='200 MiB') \
    .to_parquet(f'{dir02}dd_base_interm.parquet')

# remove intermediate files
for latmin, latmax in zip(list(range(70, -61, -10)), 
                          list(range(80, -51, -10))):    
    file = (f'dd_base_latmin{latmin}_latmax{latmax}.parquet')

    if os.path.exists(dir02 + file):
        shutil.rmtree(dir02 + file)

---

### Select training data and create database

In [ ]:
#  Get intermediate data
dd_base = dd.read_parquet(f'{dir02}dd_base_interm.parquet')

In [ ]:
def add_train_test_cols_01degree(ctype, etype):
    """
    Selects training and testing data and adds respective rows to dd_base.
    Training and testing data are selected by choosing the grid cell 
    with the highest carbon value within each 0.1-degree tile (if available).
    """

    if etype in ['trop_ef', 'temp_dbf', 'temp_enf', 'temp_dnf', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']:
        ncells = 1
    if etype in ['trop_df', 'temp_ebf', 'glob_shrub', 'glob_grass']:
        ncells = 2
    
    # Select carbon data for ctype + etype combination 
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
        cbase = 'csoil_000_100'
    
    # Load data in memory
    global dd_base
    dd_base = dd_base.persist()
    
    # Create dataframe with potential training data of etype
    dd_sel = dd_base[dd_base['pot_train_' + etype] == True] \
        [['lat', 'lon', f'pot_train_{etype}', cbase]]
    
    # Create 0.1° tile identifiers
    dd_sel['tile_lat_0_1'] = (dd_sel['lat'] * 10).astype(int) / 10.0
    dd_sel['tile_lon_0_1'] = (dd_sel['lon'] * 10).astype(int) / 10.0
    
    # Group by 0.1° tiles and select the grid cell with the highest value
    dd_sel_top = dd_sel.compute() \
        .groupby(['tile_lat_0_1', 'tile_lon_0_1']) \
        .apply(lambda x: x.nlargest(ncells, cbase), include_groups=False) \
        .reset_index()[['lat', 'lon']]
    
    # From selected rows generate training and testing dataframes
    # Note: May cause spatial autocorrelation
    dd_train, dd_test = train_test_split(
        dd_sel_top, test_size=0.20, random_state=42)

    # Add train and test columns to respective dataframes
    dd_train[f'train_{ctype}_{etype}'] = True
    dd_test[f'test_{ctype}_{etype}'] = True

    # Merge dataframes with dd_base to add training and testing columns
    dd_base = dd_base \
        .merge(dd_train, how='left', on=['lat', 'lon']) \
        .merge(dd_test, how='left', on=['lat', 'lon'])

In [ ]:
# Loop through all combinations of ctype and etype
for ctype in ['cveg', 'csoil']:
    for etype in list_etype:
        add_train_test_cols_01degree(ctype, etype)

In [ ]:
# Select and order relevant columns
dd_base = dd_base[
    ['lat', 'lon', 'copernicus_land_mask',
     'cveg_biom',  'cveg_tund', 'cveg_crop', 'cveg_gras', 'csoil_000_100',
     *[f'pot_occur_{i}' for i in list_etype], 
     *[f'pot_train_{i}' for i in list_etype],
     *[f'train_cveg_{i}' for i in list_etype],
     *[f'train_csoil_{i}' for i in list_etype],
     *[f'test_cveg_{i}' for i in list_etype], 
     *[f'test_csoil_{i}' for i in list_etype],
     *list_var_exp]]

# Remove database file if exists
if os.path.exists(f'{dir02}dd_base_r001.parquet'):
    shutil.rmtree(f'{dir02}dd_base_r001.parquet')
    
# Export
dd_base.to_parquet(f'{dir02}dd_base_r001.parquet')

# Remove intermediate files
for i in [i for i in os.listdir(dir02) if i.endswith('interm.parquet')]:
    shutil.rmtree(f'{dir02}{i}')

In [ ]:
cluster.close()